### Setup
Loading the libraries we need and the company universe: 35 U.S. retailers, 
covering 2019–2023 10-K filings.

In [16]:
import pandas as pd
import numpy as np
import os
import re
import sys
import time
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

sys.path.append("../src")
from extract_risk_section import extract_risk_factors

company_df = pd.read_csv("../data/processed/company_universe.csv")
print(f"Total companies: {len(company_df)}")
company_df.head()

Total companies: 35


,ticker,company_name
0,WMT,Walmart
1,TGT,Target
2,COST,Costco
3,BJ,BJ's Wholesale
4,HD,Home Depot


### Mapping companies to their folder names
A few companies were downloaded under their CIK number instead of ticker 
symbol earlier, so we have to build a mapping that handles both cases 
before looping through everyone consistently.

In [17]:
sec_filings_root = "../data/raw/sec_filings/sec-edgar-filings"
downloaded_folders = sorted(os.listdir(sec_filings_root))

folder_overrides = {
    "0000039911": "GPS",
    "0000072333": "JWN",
    "0001618921": "WBA",
}

ticker_to_folder = {}
for folder in downloaded_folders:
    if folder in folder_overrides:
        ticker_to_folder[folder_overrides[folder]] = folder
    else:
        ticker_to_folder[folder] = folder

print(f"Total mapped: {len(ticker_to_folder)}")
missing = set(company_df["ticker"]) - set(ticker_to_folder.keys())
print(f"Missing from mapping: {missing}")

Total mapped: 35
Missing from mapping: set()


### Extracting risk sections for all companies
Looping through every company's filings and pulling out the Risk Factors 
section. We isolate the actual 10-K document first (filings bundle in a lot 
of exhibits we don't need), then find the real section header by checking 
for the words "Risk Factors" nearby, skipping comma-style cross-references, 
and picking whichever match has the largest gap before the next section.

In [36]:
results = []
extraction_failures = []

for ticker, folder_name in ticker_to_folder.items():
    company_folder = os.path.join(sec_filings_root, folder_name, "10-K")

    if not os.path.exists(company_folder):
        extraction_failures.append((ticker, "no 10-K folder found"))
        continue

    for accession_folder in sorted(os.listdir(company_folder)):
        filing_path = os.path.join(company_folder, accession_folder, "full-submission.txt")

        if not os.path.exists(filing_path):
            extraction_failures.append((ticker, f"{accession_folder}: no filing file"))
            continue

        year_code = accession_folder.split("-")[1]
        filing_year = 2000 + int(year_code)

        risk_text = extract_risk_factors(filing_path)

        if len(risk_text) < 1000:
            extraction_failures.append((ticker, f"{accession_folder}: only {len(risk_text)} chars extracted"))
            continue

        results.append({
            "ticker": ticker,
            "filing_year": filing_year,
            "accession_number": accession_folder,
            "risk_text": risk_text,
            "risk_text_length": len(risk_text),
        })

    print(f"{ticker}: processed")

print(f"\nTotal successful extractions: {len(results)}")
print(f"Total failures: {len(extraction_failures)}")
if extraction_failures:
    print("\nFailures:")
    for f in extraction_failures:
        print(f"  {f}")

GPS: processed
JWN: processed
WBA: processed
ACI: processed
AEO: processed
ANF: processed
AZO: processed
BBBY: processed
BBY: processed
BJ: processed
BURL: processed
CHWY: processed
COST: processed
CVNA: processed
CVS: processed
DG: processed
DKS: processed
DLTR: processed
ETSY: processed
FIVE: processed
GME: processed
HD: processed
KR: processed
KSS: processed
LOW: processed
M: processed
ORLY: processed
RH: processed
ROST: processed
TGT: processed
TJX: processed
URBN: processed
W: processed
WMT: processed
WSM: processed

Total successful extractions: 174
Total failures: 0


### Final validation
Checking the data is complete, free of duplicates, and that no company's 
text is identical year over year before treating this as final.

In [19]:
results_df = pd.DataFrame(results)

years_per_company = results_df.groupby("ticker")["filing_year"].agg(["count", "min", "max", "nunique"])
print(f"Companies with fewer than 5 years: {(years_per_company['count'] < 5).sum()}")
print(f"Companies with duplicate years: {(years_per_company['count'] != years_per_company['nunique']).sum()}")

results_df_sorted = results_df.sort_values(["ticker", "filing_year"])
identical_count = 0
for ticker in results_df_sorted["ticker"].unique():
    company_rows = results_df_sorted[results_df_sorted["ticker"] == ticker].reset_index(drop=True)
    for i in range(len(company_rows) - 1):
        if company_rows.loc[i, "risk_text"] == company_rows.loc[i + 1, "risk_text"]:
            identical_count += 1
            print(f"WARNING: {ticker} identical text {company_rows.loc[i, 'filing_year']}-{company_rows.loc[i+1, 'filing_year']}")

print(f"\nIdentical year-pairs found: {identical_count}")
print(f"\nLength distribution:")
print(results_df["risk_text_length"].describe())

Companies with fewer than 5 years: 1
Companies with duplicate years: 0

Identical year-pairs found: 0

Length distribution:
count       174.000000
mean      81298.074713
std       47194.908537
min       15466.000000
25%       44316.250000
50%       69170.500000
75%      110149.500000
max      239074.000000
Name: risk_text_length, dtype: float64


### Saving the extracted data

In [38]:
results_df = pd.DataFrame(results)
results_df.to_csv("../data/processed/risk_sections_raw.csv", index=False)
print(f"Saved {len(results_df):,} rows")

Saved 174 rows


In [35]:
import sys
if "extract_risk_section" in sys.modules:
    del sys.modules["extract_risk_section"]
from extract_risk_section import extract_risk_factors

print("Reloaded — testing on Burlington directly")
burl_dir = "../data/raw/sec_filings/sec-edgar-filings/BURL/10-K"
burl_2022_accession = [a for a in os.listdir(burl_dir) if "-22-" in a][0]
burl_2022_path = os.path.join(burl_dir, burl_2022_accession, "full-submission.txt")
burl_test = extract_risk_factors(burl_2022_path)
print(f"BURL 2022: {len(burl_test):,} characters")

Reloaded — testing on Burlington directly
BURL 2022: 77,535 characters
